# YOLO26-OBB Training — Tosano Food Labels

**Task**: Oriented Bounding Box (OBB) detection per etichette alimentari — singola classe `food_label`  
**Modello**: `yolo26s-obb.pt` pretrained su DOTAv1 → fine-tuning su ~183 immagini  
**Augmentation**: tutti i default Ultralytics (nessun override)  

| Parametro | Valore |
|-----------|--------|
| Epochs | 150 (early stop patience=50) |
| Batch | 16 |
| imgsz | 640 |
| Optimizer | auto (MuSGD — default YOLO26) |

> **Ref**: [docs.ultralytics.com/tasks/obb](https://docs.ultralytics.com/tasks/obb/) · [docs.ultralytics.com/models/yolo26](https://docs.ultralytics.com/models/yolo26/)

---

**Default augmentation** (`ultralytics/cfg/default.yaml` v8.4.x):
```
hsv_h=0.015  hsv_s=0.7   hsv_v=0.4     # colore/luminosità
translate=0.1  scale=0.5                # geometria
fliplr=0.5                              # flip orizzontale
mosaic=1.0   close_mosaic=10            # mosaic ON (ottimo per dataset piccoli)
degrees=0.0  flipud=0.0  mixup=0.0      # OFF
```


## 1. Setup Environment


In [ ]:
# Controlla GPU disponibile
!nvidia-smi

# Installa Ultralytics — YOLO26 OBB richiede >=8.4.0
!pip install -q "ultralytics>=8.4.0"

import torch
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU:  {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 8:
        print("⚠️  VRAM < 8GB: considera batch=8 anziché 16")
else:
    raise RuntimeError(
        "Nessuna GPU trovata! "
        "Vai su Runtime → Cambia tipo di runtime → GPU (T4 o superiore)"
    )


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ Aggiorna questi path con la tua struttura su Google Drive
# Il file ZIP è prodotto da: scripts/prepare_dataset_obb.py
DRIVE_ZIP_PATH = "/content/drive/MyDrive/tosano_food_labels/tosano_obb.zip"
LOCAL_DATASET  = "/content/dataset_obb"
OUTPUT_PATH    = "/content/drive/MyDrive/tosano_food_labels/models_obb"


## 3. Extract Dataset


In [ ]:
import os, shutil, zipfile

os.makedirs(LOCAL_DATASET, exist_ok=True)
os.makedirs(OUTPUT_PATH,   exist_ok=True)

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(
        f"Dataset ZIP non trovato: {DRIVE_ZIP_PATH}\n"
        "Esegui prima scripts/prepare_dataset_obb.py e carica lo ZIP su Drive."
    )

print(f"Estraendo dataset da {DRIVE_ZIP_PATH}...")
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
    z.extractall(LOCAL_DATASET)

print(f"✅ Dataset estratto in {LOCAL_DATASET}")


## 4. Verify Dataset Structure

Struttura attesa (prodotta da `scripts/prepare_dataset_obb.py`):
```
dataset_obb/
├── images/  {train/ val/ test/}
├── labels/  {train/ val/ test/}   ← formato: class x1 y1 x2 y2 x3 y3 x4 y4
└── data.yaml
```

> **Nota OBB**: coordinate normalizzate [0,1], angoli vincolati a [0°, 90°)  
> Internamente YOLO converte in formato `xywhr` (centro + dimensioni + angolo)


In [ ]:
def count_files(directory):
    if not os.path.exists(directory):
        return 0
    return len([f for f in os.listdir(directory)
                if os.path.isfile(os.path.join(directory, f))])

train_imgs = os.path.join(LOCAL_DATASET, "images", "train")
val_imgs   = os.path.join(LOCAL_DATASET, "images", "val")
test_imgs  = os.path.join(LOCAL_DATASET, "images", "test")
train_lbls = os.path.join(LOCAL_DATASET, "labels", "train")
val_lbls   = os.path.join(LOCAL_DATASET, "labels", "val")
test_lbls  = os.path.join(LOCAL_DATASET, "labels", "test")
data_yaml  = os.path.join(LOCAL_DATASET, "data.yaml")

n_train = count_files(train_imgs)
n_val   = count_files(val_imgs)
n_test  = count_files(test_imgs)

print("📊 Dataset OBB — riepilogo:")
print(f"  Train: {n_train} immagini, {count_files(train_lbls)} label")
print(f"  Val:   {n_val}   immagini, {count_files(val_lbls)} label")
print(f"  Test:  {n_test}  immagini, {count_files(test_lbls)} label")
print(f"  Totale: {n_train + n_val + n_test} immagini")

if n_train == 0:
    raise ValueError("Nessuna immagine di training trovata! Controlla DRIVE_ZIP_PATH.")

# Mostra un label di esempio
for f in os.listdir(train_lbls):
    if f.endswith('.txt'):
        with open(os.path.join(train_lbls, f)) as fp:
            content = fp.read().strip()
        print(f"\n📄 Esempio label OBB ({f}):")
        print(f"  '{content[:120]}'")
        print(f"  Token: {len(content.split())} (attesi: 9 = class + 4×2 coordinate)")
        break

# Aggiorna data.yaml con il path Colab corretto
data_yaml_content = f"""path: {LOCAL_DATASET}
train: images/train
val:   images/val
test:  images/test

names:
  0: food_label
"""
with open(data_yaml, "w") as f:
    f.write(data_yaml_content)
print(f"\n📄 data.yaml aggiornato")


## 5. Load Pretrained YOLO26-OBB Model

| Modello | Params | GFLOPs | Note |
|---------|--------|--------|------|
| yolo26n-obb.pt | 2.7M | 16.9 | Nano |
| **yolo26s-obb.pt** | **10.6M** | **63.5** | **Small ← usato** |
| yolo26m-obb.pt | 23.6M | 211.9 | Medium |
| yolo26l-obb.pt | 28.0M | 259.0 | Large |
| yolo26x-obb.pt | 62.8M | 578.9 | XL |

Pretraining: **DOTAv1** (15 classi — aerei, navi, veicoli da satellite)  
I pesi del backbone si adattano via transfer learning alla detection di etichette.


In [ ]:
# Download automatico da GitHub ultralytics/assets/releases
MODEL_NAME = "yolo26s-obb.pt"

model = YOLO(MODEL_NAME)

print(f"✅ Modello caricato: {MODEL_NAME}")
print(f"   Task:   {model.task}")
print(f"   Classi: {model.names}")


## 6. Training

**Strategia**: nessun override di augmentation → usa tutti i default YOLO26.  
Il `mosaic=1.0` (default) è particolarmente efficace con dataset piccoli (~128 img train).  
YOLO26 aggiunge automaticamente la **angle loss** per OBB (risolve discontinuità a 90°).


In [ ]:
# Parametri strutturali — l'unica cosa che personalizziamo
EPOCHS     = 150   # +50 rispetto al default 100 (dataset piccolo)
BATCH_SIZE = 16    # Riduci a 8 se OOM su T4
IMAGE_SIZE = 640   # Default Ultralytics
PATIENCE   = 50    # Early stop senza miglioramento

print("=" * 65)
print("CONFIGURAZIONE TRAINING YOLO26-OBB")
print("=" * 65)
print(f"  Modello:       {MODEL_NAME}")
print(f"  Task:          OBB (Oriented Bounding Box)")
print(f"  Classe:        food_label (1 classe)")
print(f"  Image Size:    {IMAGE_SIZE}px")
print(f"  Epochs:        {EPOCHS} (early stop patience={PATIENCE})")
print(f"  Batch:         {BATCH_SIZE}")
print(f"  Augmentation:  TUTTI DEFAULT Ultralytics YOLO26")
print(f"  Optimizer:     auto (MuSGD — default per YOLO26)")
print("=" * 65)

# model.train() con *-obb.pt imposta automaticamente task='obb'
# NON serve specificare task= esplicitamente
# NON sono specificati parametri di augmentation → usa default.yaml
results = model.train(
    data=data_yaml,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    patience=PATIENCE,
    optimizer="auto",          # MuSGD (ibrido SGD+Muon, default YOLO26)
    project=OUTPUT_PATH,
    name="tosano_obb",
    exist_ok=True,
    save=True,
    save_period=25,            # checkpoint ogni 25 epoch
    plots=True,
    amp=True,                  # Automatic Mixed Precision
)

print("\n✅ Training completato!")


## 7. Evaluate on Validation Set

Le metriche OBB usano la stessa API della detection:  
`metrics.box.map50` · `metrics.box.map` · `metrics.box.mp` · `metrics.box.mr`

> Ref: [docs.ultralytics.com/tasks/obb/#val](https://docs.ultralytics.com/tasks/obb/#val)


In [ ]:
from IPython.display import Image, display

print("\n⏳ Validazione in corso...")
metrics = model.val()

print(f"\n📊 Risultati Validazione:")
print(f"  mAP50:     {metrics.box.map50:.4f}")
print(f"  mAP50-95:  {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall:    {metrics.box.mr:.4f}")

results_path = os.path.join(OUTPUT_PATH, "tosano_obb")

for plot_name, title in [
    ("results.png",           "Curve di training"),
    ("confusion_matrix.png",  "Confusion Matrix"),
    ("val_batch0_pred.jpg",   "Esempi predizione (val batch 0)"),
]:
    p = os.path.join(results_path, plot_name)
    if os.path.exists(p):
        print(f"\n{title}:")
        display(Image(filename=p))


## 8. Visual Test su Immagini di Validazione

Per le predizioni OBB usa `result.obb` (non `result.boxes`):

| Attributo | Contenuto |
|-----------|-----------|
| `result.obb.xywhr` | centro-x, centro-y, w, h, angolo (radianti) |
| `result.obb.xyxyxyxy` | 4 angoli del rettangolo orientato |
| `result.obb.conf` | confidenza |
| `result.obb.cls` | classe |

> Ref: [docs.ultralytics.com/tasks/obb/#predict](https://docs.ultralytics.com/tasks/obb/#predict)


In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

best_weights = os.path.join(results_path, "weights", "best.pt")
if not os.path.exists(best_weights):
    best_weights = os.path.join(results_path, "weights", "last.pt")
    print("⚠️  best.pt non trovato, uso last.pt")

model_best = YOLO(best_weights)
print(f"\n✅ Pesi migliori caricati: {best_weights}")

val_image_files = [
    f for f in os.listdir(val_imgs)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

if val_image_files:
    n_show = min(4, len(val_image_files))
    fig, axes = plt.subplots(1, n_show, figsize=(7 * n_show, 8))
    if n_show == 1:
        axes = [axes]

    for i in range(n_show):
        img_path = os.path.join(val_imgs, val_image_files[i])
        preds = model_best(img_path, verbose=False)

        axes[i].imshow(cv2.cvtColor(preds[0].plot(), cv2.COLOR_BGR2RGB))
        axes[i].set_title(val_image_files[i], fontsize=9)
        axes[i].axis("off")

        if preds[0].obb is not None and len(preds[0].obb) > 0:
            for obb_box in preds[0].obb:
                xywhr = obb_box.xywhr[0].cpu().numpy()
                conf  = float(obb_box.conf[0])
                angle_deg = float(xywhr[4]) * 180 / np.pi
                print(f"  {val_image_files[i]}: "
                      f"cx={xywhr[0]:.1f} cy={xywhr[1]:.1f} "
                      f"w={xywhr[2]:.1f} h={xywhr[3]:.1f} "
                      f"angle={angle_deg:.1f}° conf={conf:.3f}")
        else:
            print(f"  {val_image_files[i]}: nessuna detection")

    plt.tight_layout()
    plt.show()


## 9. Test Set Evaluation (Holdout)


In [ ]:
if n_test > 0:
    print("\n" + "=" * 65)
    print("VALUTAZIONE SUL TEST SET (holdout)")
    print("=" * 65)

    test_metrics = model_best.val(data=data_yaml, split="test", verbose=False)

    print(f"\n📊 Risultati Test Set:")
    print(f"  mAP50:     {test_metrics.box.map50:.4f}")
    print(f"  mAP50-95:  {test_metrics.box.map:.4f}")
    print(f"  Precision: {test_metrics.box.mp:.4f}")
    print(f"  Recall:    {test_metrics.box.mr:.4f}")
else:
    print("⚠️  Test set vuoto — salta valutazione holdout.")


## 10. Save & Export Model

Copia `best.pt` su Drive con nome descrittivo.
Scommentare per esportare in ONNX.


In [ ]:
final_pt_path = os.path.join(OUTPUT_PATH, "tosano-yolo26s-obb.pt")
shutil.copy(best_weights, final_pt_path)

print(f"\n✅ Modello salvato: {final_pt_path}")
print("""
Uso nel pipeline Tosano:

  from ultralytics import YOLO
  import numpy as np

  model = YOLO('tosano-yolo26s-obb.pt')
  results = model('image.jpg')

  for result in results:
      if result.obb is not None and len(result.obb) > 0:
          xyxyxyxy = result.obb.xyxyxyxy.cpu().numpy()  # (N, 4, 2)
          xywhr    = result.obb.xywhr.cpu().numpy()      # (N, 5)
          confs    = result.obb.conf.cpu().numpy()
""")

# Export ONNX (opzionale)
# model_best.export(format="onnx")


## 11. Download Model


In [ ]:
from google.colab import files

print("\n⬇️  Download modello...")
try:
    files.download(final_pt_path)
    print("✅ Download avviato!")
except Exception:
    print(f"Download automatico fallito.\nScarica manualmente da Drive: {final_pt_path}")
